In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"P:\AI\Project_Default\data\raw\accepted_2007_to_2018Q4_xl.csv")

C:\Users\yaswa\AppData\Local\Temp\ipykernel_11224\902509769.py:1: DtypeWarning: Columns (0: id, 1: desc, 2: next_pymnt_d, 3: verification_status_joint, 4: sec_app_earliest_cr_line, 5: hardship_type, 6: hardship_reason, 7: hardship_status, 8: hardship_start_date, 9: hardship_end_date, 10: payment_plan_start_date, 11: hardship_loan_status, 12: debt_settlement_flag_date, 13: settlement_status, 14: settlement_date) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"P:\AI\Project_Default\data\raw\accepted_2007_to_2018Q4_xl.csv")


In [3]:
df.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df.shape

(2260701, 151)

In [5]:
# Create a summary dataframe of your columns
df_info = pd.DataFrame({
    'Column Name': df.columns,
    'Data Type': df.dtypes.values,
    'Missing Values': df.isnull().sum().values,
    'Percentage Missing': (df.isnull().sum().values / len(df)) * 100
})

In [6]:
import pandas as pd
pd.set_option('display.max_rows', None)
print(df_info.sort_values(by='Percentage Missing', ascending=False).to_string())

                                    Column Name Data Type  Missing Values  Percentage Missing
1                                     member_id   float64         2260701          100.000000
140  orig_projected_additional_accrued_interest   float64         2252050           99.617331
130                             hardship_reason       str         2249784           99.517097
141              hardship_payoff_balance_amount   float64         2249784           99.517097
142                hardship_last_payment_amount   float64         2249784           99.517097
136                     payment_plan_start_date       str         2249784           99.517097
129                               hardship_type       str         2249784           99.517097
131                             hardship_status       str         2249784           99.517097
134                         hardship_start_date       str         2249784           99.517097
132                               deferral_term   float64   

In [ ]:
# Step 1 - Drop columns per notes/01_eda.md (Reason 1: too many nulls, Reason 2: not useful regardless of missing %)

# Reason 1a - >=70% missing, hardship/settlement/joint-applicant fields (deferred to survival/uplift analysis later)
drop_high_missing = [
    'member_id', 'orig_projected_additional_accrued_interest', 'hardship_reason',
    'hardship_payoff_balance_amount', 'hardship_last_payment_amount', 'payment_plan_start_date',
    'hardship_type', 'hardship_status', 'hardship_start_date', 'deferral_term', 'hardship_amount',
    'hardship_dpd', 'hardship_loan_status', 'hardship_length', 'hardship_end_date',
    'settlement_status', 'debt_settlement_flag_date', 'settlement_term', 'settlement_percentage',
    'settlement_date', 'settlement_amount', 'sec_app_mths_since_last_major_derog',
    'sec_app_revol_util', 'revol_bal_joint', 'sec_app_inq_last_6mths', 'sec_app_num_rev_accts',
    'sec_app_open_acc', 'sec_app_earliest_cr_line', 'sec_app_fico_range_high', 'sec_app_mort_acc',
    'sec_app_open_act_il', 'sec_app_fico_range_low', 'sec_app_collections_12_mths_ex_med',
    'sec_app_chargeoff_within_12_mths', 'verification_status_joint', 'dti_joint',
    'annual_inc_joint', 'desc', 'mths_since_last_record', 'mths_since_recent_bc_dlq',
    'mths_since_last_major_derog',
]

# Reason 1b - 38.3% missing, credit-bureau snapshot fields (data-collection-era artifact, overlaps Bucket 4)
drop_38pct_band = [
    'open_acc_6m', 'inq_last_12m', 'total_cu_tl', 'open_il_24m', 'max_bal_bc', 'open_act_il',
    'open_il_12m', 'open_rv_24m', 'total_bal_il', 'inq_fi', 'open_rv_12m',
]

# Reason 2 - not useful regardless of missing % (identifiers, near-constant, or redundant)
drop_not_useful = [
    'title', 'id', 'url', 'policy_code', 'zip_code', 'pymnt_plan',
    'initial_list_status', 'disbursement_method',
]

# 40-70% missing band:
# - mths_since_recent_revol_delinq, il_util, mths_since_rcnt_il, all_util: credit bureau snapshot
#   fields not needed for the initial model, high missingness with no unique signal beyond what's
#   already kept.
# - next_pymnt_d: dropped for a different reason - it's structurally near-always null for closed
#   loans (no "next" payment once a loan has ended), so its missingness isn't informative for a
#   classifier restricted to already-concluded loans; the rare non-null values would also leak
#   information about loan status at prediction time.
drop_40_70pct_band = [
    'mths_since_recent_revol_delinq', 'next_pymnt_d', 'il_util', 'mths_since_rcnt_il', 'all_util',
]

cols_to_drop = drop_high_missing + drop_38pct_band + drop_not_useful + drop_40_70pct_band
df = df.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} columns. Remaining shape: {df.shape}")


In [8]:
df['loan_status'].value_counts()

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64

In [9]:
# Apply Bucket 1 label mapping (see data_dictionary.md) to check class split
label_map = {
    'Fully Paid': 0,
    'Current': 'CENSOR',
    'Charged Off': 1,
    'Late (31-120 days)': 1,
    'In Grace Period': 0,
    'Late (16-30 days)': 0,
    'Does not meet the credit policy. Status:Fully Paid': 0,
    'Does not meet the credit policy. Status:Charged Off': 1,
    'Default': 1,
}

df['target'] = df['loan_status'].map(label_map)

total = len(df)
group_counts = df['target'].value_counts(dropna=False)
group_pct = (group_counts / total * 100).round(2)

print(f"Total rows: {total}\n")
print("--- Full dataset grouping ---")
for label in group_counts.index:
    print(f"{label}: {group_counts[label]} ({group_pct[label]}%)")

# Classifier-eligible subset excludes censored 'Current' rows
clf_df = df[df['target'] != 'CENSOR']
clf_counts = clf_df['target'].value_counts()
clf_pct = (clf_counts / len(clf_df) * 100).round(2)

print(f"\n--- Initial classifier subset (excludes Current) ---")
print(f"Classifier rows: {len(clf_df)} ({len(clf_df)/total*100:.2f}% of full data)")
print(f"Class 0 (Did not default): {clf_counts[0]} ({clf_pct[0]}%)")
print(f"Class 1 (Defaulted): {clf_counts[1]} ({clf_pct[1]}%)")
print(f"Imbalance ratio: 1:{clf_counts[0]/clf_counts[1]:.2f}")


Total rows: 2260701

--- Full dataset grouping ---
0: 1091524 (48.28%)
CENSOR: 878317 (38.85%)
1: 290827 (12.86%)
nan: 33 (0.0%)

--- Initial classifier subset (excludes Current) ---
Classifier rows: 1382384 (61.15% of full data)
Class 0 (Did not default): 1091524 (78.96%)
Class 1 (Defaulted): 290827 (21.04%)
Imbalance ratio: 1:3.75


In [ ]:
# Step 3 - Missing value imputation on clf_df, per decisions documented in notes/01_eda.md
clf_df = clf_df.copy()

# "Absence is informative" columns - sentinel fill + explicit flag
sentinel = 999
for col in ['mths_since_last_delinq', 'mths_since_recent_inq', 'num_tl_120dpd_2m', 'mo_sin_old_il_acct']:
    clf_df[f'{col}_missing_flag'] = clf_df[col].isnull().astype(int)
    clf_df[col] = clf_df[col].fillna(sentinel)

# emp_length - unknown category
clf_df['emp_length'] = clf_df['emp_length'].fillna('unknown')

# Credit-bureau summary-stat cluster (~35 cols, ~2.2-3.4% missing) - zero fill,
# absence of relevant trade lines naturally implies 0 for counts/balances/utilization
zero_fill_cols = [
    'bc_util', 'percent_bc_gt_75', 'bc_open_to_buy', 'pct_tl_nvr_dlq', 'total_rev_hi_lim',
    'tot_hi_cred_lim', 'total_il_high_credit_limit', 'total_bal_ex_mort', 'total_bc_limit',
    'avg_cur_bal', 'tot_cur_bal', 'tot_coll_amt', 'mths_since_recent_bc', 'mo_sin_rcnt_rev_tl_op',
    'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_tl', 'num_rev_accts', 'num_op_rev_tl', 'num_il_tl',
    'num_actv_rev_tl', 'num_actv_bc_tl', 'num_bc_tl', 'num_bc_sats', 'num_sats',
    'num_rev_tl_bal_gt_0', 'acc_open_past_24mths', 'mort_acc', 'num_tl_op_past_12m',
    'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_accts_ever_120_pd',
]
clf_df[zero_fill_cols] = clf_df[zero_fill_cols].fillna(0)

# emp_title - dropped for now (messy free text, not usable without NLP grouping)
clf_df = clf_df.drop(columns=['emp_title'])

# Remaining negligible-missing columns (<=0.18% each, at most a couple thousand out of 1.38M rows,
# largely overlapping bad/incomplete CSV rows) - safe to drop the rows rather than impute.
# Verified none of these are high-missing columns before including them here (see notes/01_eda.md
# note on all_util/next_pymnt_d, which were moved to the Step 1 column-drop list instead).
row_drop_cols = [
    'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade',
    'sub_grade', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status',
    'purpose', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low',
    'fico_range_high', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util',
    'total_acc', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv',
    'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries',
    'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'last_credit_pull_d',
    'last_fico_range_high', 'last_fico_range_low', 'application_type', 'acc_now_delinq',
    'chargeoff_within_12_mths', 'delinq_amnt', 'pub_rec_bankruptcies', 'tax_liens',
    'hardship_flag', 'debt_settlement_flag', 'target', 'collections_12_mths_ex_med',
]
clf_df = clf_df.dropna(subset=row_drop_cols)

print(f"Shape after Step 3: {clf_df.shape}")
remaining_nulls = clf_df.isnull().sum()
print("Columns still containing nulls after Step 3:")
print(remaining_nulls[remaining_nulls > 0])


### Step 4 - Leakage audit (see notes/01_eda.md "Step 4: Leakage audit")

Before using any payment/credit fields as classifier features, exclude columns only known
*after* a loan concludes - they encode the outcome itself rather than predicting it.

First pass was driven by the "remaining nulls after Step 3" list, which misses zero-null
columns. Did a full pass over all 84 remaining feature-candidate columns instead.

**Leaky (exclude from classifier feature set):**
`recoveries`, `collection_recovery_fee`, `total_pymnt`, `total_pymnt_inv`, `total_rec_prncp`,
`total_rec_int`, `total_rec_late_fee`, `out_prncp`, `out_prncp_inv`, `last_pymnt_amnt`,
`last_pymnt_d`, `last_fico_range_low`, `last_fico_range_high`, `last_credit_pull_d`,
`hardship_flag`, `debt_settlement_flag` (mid-loan distress events), `funded_amnt_inv`
(redundant with `loan_amnt`, use that instead)

**Borderline (bureau-pull-time snapshots, believed safe but not yet empirically verified)** -
checked in the next cell by comparing distributions across target classes:
`chargeoff_within_12_mths`, `collections_12_mths_ex_med`, `acc_now_delinq`, `delinq_amnt`,
`total_rev_hi_lim`, `tot_hi_cred_lim`, `tot_cur_bal`, `total_bal_ex_mort`, `total_bc_limit`,
`total_il_high_credit_limit`, `avg_cur_bal`, `bc_open_to_buy`, `bc_util`, `percent_bc_gt_75`,
`pct_tl_nvr_dlq`, `num_accts_ever_120_pd`, `num_actv_bc_tl`, `num_actv_rev_tl`, `num_bc_sats`,
`num_bc_tl`, `num_il_tl`, `num_op_rev_tl`, `num_rev_accts`, `num_rev_tl_bal_gt_0`, `num_sats`,
`num_tl_30dpd`, `num_tl_90g_dpd_24m`, `num_tl_op_past_12m`, `mo_sin_old_rev_tl_op`,
`mo_sin_rcnt_rev_tl_op`, `mo_sin_rcnt_tl`, `mort_acc`, `mths_since_recent_bc`,
`mths_since_recent_inq`, `acc_open_past_24mths`

These stay in `clf_df` (may be useful later, e.g. survival/uplift analysis), but leaky columns
must not be fed into the initial classifier as features.


In [11]:
# Step 4 - Empirical leakage check on borderline bureau-snapshot columns.
# Compare distribution (mean, median, std) by target class - a column that looks
# suspiciously different across classes in a way that doesn't make domain sense
# warrants a closer look before being trusted as safe.
borderline_cols = [
    'chargeoff_within_12_mths', 'collections_12_mths_ex_med', 'acc_now_delinq', 'delinq_amnt',
    'total_rev_hi_lim', 'tot_hi_cred_lim', 'tot_cur_bal', 'total_bal_ex_mort', 'total_bc_limit',
    'total_il_high_credit_limit', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'percent_bc_gt_75',
    'pct_tl_nvr_dlq', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_sats',
    'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats',
    'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m', 'mo_sin_old_rev_tl_op',
    'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc', 'mths_since_recent_bc',
    'mths_since_recent_inq', 'acc_open_past_24mths',
]

leakage_check = clf_df.groupby('target')[borderline_cols].agg(['mean', 'median', 'std']).T
leakage_check.columns = ['target_0', 'target_1']
leakage_check['ratio'] = leakage_check['target_1'] / leakage_check['target_0'].replace(0, pd.NA)
print(leakage_check.to_string())


                                        target_0       target_1     ratio
chargeoff_within_12_mths   mean         0.008893       0.009807   1.10276
                           median       0.000000       0.000000      <NA>
                           std          0.108850       0.113090  1.038953
collections_12_mths_ex_med mean         0.016036       0.022018  1.373031
                           median       0.000000       0.000000      <NA>
                           std          0.143141       0.162560  1.135663
acc_now_delinq             mean         0.004881       0.005525   1.13211
                           median       0.000000       0.000000      <NA>
                           std          0.075785       0.080810  1.066309
delinq_amnt                mean        13.701000      19.107222  1.394586
                           median       0.000000       0.000000      <NA>
                           std        759.461710     971.213897  1.278819
total_rev_hi_lim           mean     31

In [12]:
# Sanity check / positive control: run the same check on columns we're confident ARE leaky.
# If the method works, these should show extreme ratios - unlike the borderline batch above.
known_leaky_cols = [
    'total_pymnt', 'total_rec_prncp', 'out_prncp', 'recoveries', 'collection_recovery_fee',
    'last_pymnt_amnt',
]

leaky_check = clf_df.groupby('target')[known_leaky_cols].agg(['mean', 'median', 'std']).T
leaky_check.columns = ['target_0', 'target_1']
leaky_check['ratio'] = leaky_check['target_1'] / leaky_check['target_0'].replace(0, pd.NA)
print(leaky_check.to_string())


                                    target_0     target_1     ratio
total_pymnt             mean    16378.004028  8389.809056  0.512261
                        median  13765.081394  6575.055000  0.477662
                        std     10461.457426  6750.588200  0.645282
total_rec_prncp         mean    14034.217963  4511.080933  0.321434
                        median  12000.000000  3240.460000  0.270038
                        std      8673.561755  4213.408517  0.485776
out_prncp               mean      128.219582   842.509142  6.570831
                        median      0.000000     0.000000      <NA>
                        std      1490.461570  3757.626688  2.521116
recoveries              mean        0.000000  1117.285129      <NA>
                        median      0.000000   452.485000      <NA>
                        std         0.000000  1799.353703      <NA>
collection_recovery_fee mean        0.000000   186.407123      <NA>
                        median      0.000000    

This shows the columns with ratio between ~0.7-1.4 are non-leaky, while those with ratio <0.7 or >1.4 are leaky. The leaky columns are in the output for above code snippet. 

In [ ]:
# Step 5 - Train/val/test split (70/15/15), random stratified on target (see notes/01_eda.md)
from sklearn.model_selection import train_test_split

leaky_cols = [
    'recoveries', 'collection_recovery_fee', 'total_pymnt', 'total_pymnt_inv',
    'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'out_prncp', 'out_prncp_inv',
    'last_pymnt_amnt', 'last_pymnt_d', 'last_fico_range_low', 'last_fico_range_high',
    'last_credit_pull_d', 'hardship_flag', 'debt_settlement_flag', 'funded_amnt_inv',
]
non_feature_cols = ['loan_status', 'target'] + leaky_cols

X = clf_df.drop(columns=non_feature_cols)
y = clf_df['target'].astype(int)

# First split off test (15%), then split remaining 85% into train (70% of total) and val (15% of total)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
for name, y_split in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    pct = y_split.value_counts(normalize=True) * 100
    print(f"{name} class balance -> 0: {pct[0]:.2f}%, 1: {pct[1]:.2f}%")


In [13]:
# Save splits to data/processed/ so 02_feat_engineer.ipynb can pick up from here
processed_dir = r"P:\AI\Project_Default\data\processed"

X_train.assign(target=y_train).to_parquet(f"{processed_dir}\\train.parquet", index=False)
X_val.assign(target=y_val).to_parquet(f"{processed_dir}\\val.parquet", index=False)
X_test.assign(target=y_test).to_parquet(f"{processed_dir}\\test.parquet", index=False)

print("Saved train/val/test to data/processed/")


Saved train/val/test to data/processed/
